# Training on the archaeoner data with Gliner2



### Problem: Domain specific labels are needed

- In specialized domains, additional labels may be needed to capture
domain-specific entities.
    - Biomedical domain: `Gene/Protein`, `Disease`, `Chemical`, `Drug`, etc.
    - Financial domain: `Financial Instrument`, `Market Index`, `Economic Indicator`, etc



## [GLiNER](https://github.com/fastino-ai/GLiNER2/): From Classification to Matching

- Traditional NER models function as token classifiers
    - trained to recognize a fixed set of categories
    - failing if there is a need to find `Medical Procedure` or `Programming Language` and the model wasn't trained on those specifically
- GLiNER treats entity extraction as a matching problem between the text and the entity labels (and their definitions)
    - Zero-shot model that can identify any entity type defined as a entity label at runtime
    - Uses a bidirectional transformer to encode both the input text and the entity labels
    - Span-Based Processing: Instead of labeling individual tokens, GLiNER evaluates entire "spans" of text
    - Calculates a similarity between the representation of a text span and the representation of a label. If the similarity is high, the span is tagged with that label


### GliNER2

- Unifies named entity recognition, text classification, relation extraction, and structured data extraction into a single framework
- GliNER2 uses a schema-driven approach, which allows you to define extraction requirements declaratively and execute multiple tasks in a single inference call
- CPU-efficient, making it a useful solution for transforming messy, unstructured text into clean data without the overhead of a large language model

```python
# Use create_schema() for multi-task scenarios
schema = (extractor.create_schema()
    # Extract key entities
    .entities({
        "person": "Names of people, executives, or individuals",
        "company": "Organization, corporation, or business names",
        "product": "Products, services, or offerings mentioned"
    })
    
    # Classify the content
    .classification("sentiment", ["positive", "negative", "neutral"])
    .classification("category", ["technology", "business", "finance", "healthcare"])
    
    # Extract relationships
    .relations(["works_for", "founded", "located_in"])
    
    # Extract structured product details
    .structure("product_info")
        .field("name", dtype="str")
        .field("price", dtype="str")
        .field("features", dtype="list")
        .field("availability", dtype="str", choices=["in_stock", "pre_order", "sold_out"])
)
```


```python

# Comprehensive extraction in one pass
text = "Apple CEO Tim Cook unveiled the revolutionary iPhone 15 Pro for $999. The device features an A17 Pro chip and titanium design. Tim Cook works for Apple, which is located in Cupertino."

results = extractor.extract(text, schema)
# Output: {
#     'entities': {
#         'person': ['Tim Cook'],
#         'company': ['Apple'],
#         'product': ['iPhone 15 Pro']
#     },
#     'sentiment': 'positive',
#     'category': 'technology',
#     'relation_extraction': {
#         'works_for': [('Tim Cook', 'Apple')],
#         'located_in': [('Apple', 'Cupertino')]
#     },
#     'product_info': [{
#         'name': 'iPhone 15 Pro',
#         'price': '$999',
#         'features': ['A17 Pro chip', 'titanium design'],
#         'availability': 'in_stock'
#     }]
# }
```


# NER with GLiNER lab

## Install prerequisite libraries






In [ ]:
# !pip install -q gliner argilla pathlib pandas gliner2
!pip install -q gliner2 argilla

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.1 MB/s eta 0:00:00


In [ ]:
import io
import os
import json
import logging
import textwrap
import sys
from pathlib import Path
from getpass import getpass
from typing import List, Dict, Any, Tuple, Optional, Union
from collections import defaultdict
from itertools import combinations

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

import argilla as rg
import requests
from google import genai
from google.colab import drive
from google.colab import userdata

from gliner2 import GLiNER2

!wget -q -O ner_utils.py https://raw.githubusercontent.com/prokopidis/genai-utils/main/ner_utils.py

import importlib
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
import ner_utils
importlib.reload(ner_utils)


# Configure logging with timestamps
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,
    force=True
)

# Silence most logs for the argilla library and the underlying httpx library
logging.getLogger("argilla").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)


## Load a GLiNER2 model

In [ ]:
extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")


config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first


model.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

### A first test

In [ ]:
# # Extract entities in one line
# text = "Apple CEO Tim Cook announced iPhone 15 in Cupertino yesterday."
# text = "Ο κ. Ευάνθης Λημναίος, επικεφαλής του Ομίλου Είρωνα, ανακοίνωσε το Value 4 Family στην Αθήνα χθες."


# schema = (extractor.create_schema()
#     # Extract key entities
#     .entities({
#         "museum": "A museum is an institution dedicated to displaying and preserving culturally or scientifically significant objects.",
#         "context": "Archaeological context is the precise,3D location (provenience), surrounding material (matrix), and association with other finds that gives an object, feature, or site its meaning, date, and historical significance.",
#         "artefact": "an object formed by humans, particularly one of interest to archaeologists"
#     })

#     # # Classify the content
#     # .classification("sentiment", ["positive", "negative", "neutral"])
#     # .classification("category", ["technology", "business", "finance", "healthcare"])

#     # # Extract relationships
#     # .relations(["works_for", "founded", "located_in"])

#     # # Extract structured product details
#     # .structure("product_info")
#     #     .field("name", dtype="str")
#     #     .field("price", dtype="str")
#     #     .field("features", dtype="list")
#     #     .field("availability", dtype="str", choices=["in_stock", "pre_order", "sold_out"])
# )

# text = "Το Βρετανικό Μουσείο φιλοξενεί μία από τις Καρυάτιδες του Ερεχθείου."

# result = extractor.extract_entities(text, ["museum", "artefact", "context"])

# print(result)
# # {'entities': {'company': ['Apple'], 'person': ['Tim Cook'], 'product': ['iPhone 15'], 'location': ['Cupertino']}}

In [58]:
target_username = "stalexan" # @param {type:"string"}

# Create a reversed map from name to user_id for easy lookup
name_to_user_id = {name: user_id for user_id, name in user_id_to_name.items()}

# Get the user_id for the target_username
target_user_id = name_to_user_id.get(target_username)

if target_user_id is None:
    print(f"Error: User '{target_username}' not found in the dataset.")
else:
    # Filter completed_records for the target user
    user_specific_records = []
    for record in completed_records:
        for response in record.responses:
            if response.status == "submitted" and str(response.user_id) == target_user_id:
                user_specific_records.append(record)
                break # Move to the next record once a response from the target user is found

    print(f"Found {len(user_specific_records)} records annotated by '{target_username}'.")

    # Display a sample of the filtered records (e.g., the first 5 records)
    if user_specific_records:
        print("\nSample of filtered records:")
        for i, record in enumerate(user_specific_records[:5]): # Display first 5
            print(f"Record ID: {record.id}, Text: {record.fields.get('text', '')[:100]}...") # Truncate text for display
    else:
        print("No records found for this user.")




Found 212 records annotated by 'stalexan'.

Sample of filtered records:
Record ID: 8f5ba352-1147-4bfb-808d-fcabba63181f, Text: ...
Record ID: c8969bcc-ab3a-483c-96e1-7ab98ccb39d6, Text: ...
Record ID: 9f012a83-92a8-4254-b4f4-380541577d75, Text: ...
Record ID: 0aef1930-a74a-472c-b4bd-e66f3994ae1c, Text: ...
Record ID: d49895eb-f49a-441f-a7c7-bea74bb33d9a, Text: ...


In [59]:
if user_specific_records:
    print("\nDetailed view of keys and responses for each record:")
    for i, record in enumerate(user_specific_records):
        print(f"\n--- Record {i+1} ---")
        print(f"Record ID: {record.id}")
        print(f"Text: {record.fields.get('text', '')}")

        if record.responses:
            print("Responses:")
            for resp_idx, response in enumerate(record.responses):
                # Only display responses from the target user
                if str(response.user_id) == target_user_id:
                    print(f"  Response {resp_idx+1}:")
                    print(f"    User ID: {response.user_id}")
                    print(f"    Status: {response.status}")
                    if response.value:
                        print(f"    Value: {response.value}")
                    else:
                        print("    Value: No annotation value.")
        else:
            print("No responses for this record.")
else:
    print("No records found for the specified user.")


Detailed view of keys and responses for each record:

--- Record 1 ---
Record ID: 8f5ba352-1147-4bfb-808d-fcabba63181f
Text: 
Responses:
  Response 1:
    User ID: 521e8843-81c7-40a6-92cd-26b6b2a25544
    Status: ResponseStatus.submitted
    Value: [{'label': 'SPECIES', 'start': 16, 'end': 23}, {'label': 'SPECIES', 'start': 51, 'end': 60}]
  Response 2:
    User ID: 521e8843-81c7-40a6-92cd-26b6b2a25544
    Status: ResponseStatus.submitted
    Value: ok.

--- Record 2 ---
Record ID: c8969bcc-ab3a-483c-96e1-7ab98ccb39d6
Text: 
Responses:
  Response 1:
    User ID: 521e8843-81c7-40a6-92cd-26b6b2a25544
    Status: ResponseStatus.submitted
    Value: No annotation value.
  Response 2:
    User ID: 521e8843-81c7-40a6-92cd-26b6b2a25544
    Status: ResponseStatus.submitted
    Value: ok.

--- Record 3 ---
Record ID: 9f012a83-92a8-4254-b4f4-380541577d75
Text: 
Responses:
  Response 1:
    User ID: 521e8843-81c7-40a6-92cd-26b6b2a25544
    Status: ResponseStatus.submitted
    Value: [{'label': '

### Filter records by a specific user

Enter the username you want to filter by in the cell below. You can find available usernames in the `user_id_to_name` variable above.

### Load input data

In [ ]:
URL_TEXTS = "https://docs.google.com/spreadsheets/d/1sRjwJAD15vs5FbTaGK6x5Yc7KbjMA2jZ/edit?gid=1875924262#gid=1875924262"
URL_LABELS = "https://docs.google.com/spreadsheets/d/15MVF0kLH28_QD3ZROSpWgO0qWJiaCCxv/"

df_texts, df_labels = ner_utils.load_gsheet_data(URL_TEXTS, URL_LABELS)


2026-04-07 12:01:20 | INFO | DataFrames loaded successfully.


In [ ]:
ner_utils.display_styled_df(df_texts, n_samples=5)

,id,domain,text
10,11,archaeology,Ανασκαφή παλαιοχριστιανικού συγκροτήματος στον Λιμένα Θάσου Η βόρεια βασιλική καταλαμβάνει την αιγιαλίτιδα ζώνη και την δημοτική οδό. Είναι τρίκλιτη με νάρθηκα και ημικυκλική αψίδα. Χωρίζεται σε τρία κλίτη με κτιστούς στυλοβάτες πάνω στους οποίους εδράζονται αρχαία μαρμάρινα μέλη. Από το εσωτερικό της αποκαλύφθηκε ο κτιστός στυλοβάτης του φράγματος του πρεσβυτερίου και η βάση του άμβωνα. Στο μεσαίο κλίτος και κάτω από το δάπεδο διαμορφώνεται ταφική κρύπτη. Η νότια είναι και αυτή τρίκλιτη με νάρθηκα και ημικυκλική αψίδα. Στη νοτιοδυτική γωνία της προσκολλάται κογχωτό κτίσμα με ψηφιδωτό δάπεδο. Οι δύο βασιλικές επικοινωνούν μεταξύ τους με τρία προσκτίσματα τα δάπεδα των οποίων ακολουθούν τη διαφορά στάθμης μεταξύ των δαπέδων των δύο βασιλικών. Νότια της νότιας βασιλικής υπάρχει συγκρότημα δωματίων που περικλείουν τάφους. Σύμφωνα με τα έως τώρα δεδομένα η ύπαρξη των τάφων αποτέλεσε το έναυσμα για την δημιουργία του συγκροτήματος των βασιλικών που χρονολογούνται στον 5ο μ.Χ. αι. η νότια και 6ο μ.Χ. η βόρεια. Μετά την καταστροφή τους και στον χώρο του Ιερού Βήματος και των δύο βασιλικών ανιδρύθηκαν δύο μικρά ναΰδρια τα οποία συνέχισαν τη λατρεία στο χώρο έως και τα τέλη 12ου/αρχές 13ου αι.
2,3,archaeology,"Ενυπόγραφος λύχνος με κεφαλή Αθηνάς Λύχνος με ανάγλυφη διακόσμηση κεφαλής Αθηνάς (Προμάχου) στο δίσκο. Η θεά εικονίζεται σε κατατομή μέσα σε στεφάνη από εγχάρακτα γλωσσοειδή κοσμήματα. Στη βάση η υπογραφή ΡΗΓΛΟΥ, δηλώνει το εργαστήριο παραγωγής του λυχναριού."
14,15,archaeology,"ΑΝΑΣΚΑΦΗ ΣΤΟ ΙΕΡΟ ΤΟΥ ΑΠΟΛΛΩΝΟΣ ΑΜΥΚΛΑΙΟΥ Κατὰ μῆκος τῆς νότιας πλευρᾶς τοῦ τείχους, μπροστὰ ἀπὸ τοὺς λίθους τῆς πρόσοψής του, ἐντοπίστηκε στρῶμα ἀπὸ ἔντονα σκούρα χώματα καὶ ἴχνη καύσης, ποὺ περιεῖχε μεγάλο ἀριθμὸ μικκύλων ἀγγείων (ἔχουν καταμετρηθεῖ περίπου 2.000) καὶ μεγάλη συγκέντρωση μεταλλικῶν εὑρημάτων, τὰ ὁποῖα σύμφωνα μὲ τὴν τυπολογία τους χρονολογοῦνται στὸν 7ο καὶ 6ο αἰ. π.Χ. Ἐπὶ πλέον περισυλλέχθηκε μικρὴ ποσότητα ὀστῶν ζώων τὰ ὁποῖα φέρουν ἴχνη καύσης. Τὰ μικκύλα ἀγγεῖα ἀποτελοῦνται ἀπὸ ἀκέραιους καὶ μὴ ἀρυβάλλους λακωνικοῦ τύπου, κανθαρίσκους, ἀμφορίσκους, θυμιατήρια καὶ λάκαινες (εἰκ. 4). Ἀνάμεσα στὰ θραύσματα τῶν πήλινων εὑρημάτων ξεχωρίζουν τρία μελανόμορφα ὄστρακα (τὸ ἕνα φέρει τμῆμα μορφῆς πετεινοῦ, τὸ δεύτερο τμῆμα θαλάσσιας μορφῆς καὶ τὸ τρίτο φυτικὴ διακόσμηση ἀπὸ φύλλο κισσοῦ), τμῆμα κεφαλῆς εἰδωλίου γλαύκας τοῦ ὕστερου 6ου αἰ. π.Χ. (εἰκ. 5), κεραμίδα στέγης ποὺ φέρει ἐγχάρακτη ἐπιγραφὴ σὲ δύο σειρές, στὴ δεύτερη σειρὰ διακρίνεται τὸ ὄνομα ΗΟΡΜΙΠΠΟΣ (εἰκ. 6), εἰδώλιο βοοειδοῦς τῆς ὑστεροελλαδικῆς περιόδου καὶ τμήματα ἀπὸ ἀκροκέραμα ποὺ φέρουν μελανὸ γάνωμα καὶ χρονολογούνται στὴν ἀρχαϊκὴ περίοδο."
6,7,archaeology,"Δακτυλίδι Περιστρεφόμενη σφενδόνη δαχτυλιδιού. Ανήκει στον σκαραβοειδή τύπο, με κυρτή και επίπεδη την επιφάνεια των δύο όψεων. Στην κυρτή όψη φέρει περίτμητο ανάγλυφο χρυσό έλασμα με παράσταση Θέτιδας που μεταφέρει πάνω σε ιππόκαμπο τα όπλα του Αχιλλέα. Η ασπίδα έχει ως επίσημα το γενειοφόρο κεφάλι ανδρικής μορφής. Στην επίπεδη όψη φέρει περίτμητο ανάγλυφο χρυσό έλασμα με παράσταση γυμνού φτερωτού Έρωτα πάνω σε δελφίνι προς τα δεξιά."
3,4,archaeology,"Αναθηματική ανάγλυφη στήλη με παράσταση Κυβέλης. Έχει σχήμα ναΐσκου με παραστάδες, επιστύλιο και γείσο με ηγεμόνες καλυπτήρες κεραμίδες. Η θέα παριστάνεται ένθρονη να πατά σε υποπόδιο. Φορά ψηλά ζωσμένο χειριδωτό χιτώνα και ιμάτιο. Τμήμα του ιματίου καλύπτει τους μηρούς και τα πόδια της. Στο κεφάλι φορά κυλινδρικό πόλο. Τα μαλλιά της χωρίζονται στη μέση πάνω από το μέτωπο και πέφτουν σε μακριούς πλοκάμους πάνω στους ώμους. Με το αριστερό χέρι κρατάει τύμπανο και με το δεξί φιάλη, ενώ πάνω στα γόνατα της είναι ξαπλωμένο ένα λιονταράκι (σκύμνον)."


In [ ]:
display(df_labels.sample(n=4))

,domain,label,definition,definition_old
1,archaeology,PERIOD,"archaeological period, historical era, specific date, or century (e.g., Neolithic, 5th c. BC)","Αrchaeological periods and dates. E.g., «βυζαντινός» ναός, «προϊστορικός» οικισμός του Θορικού (“Byzantine” church; “prehistoric” settlement of Thorikos). Do not annotate modern-era periods and dates.\nExclude prefixes like “until"" (μέχρι), “about/approximately” (περίπου); annotate only the period. Annotate surrounding words for multi-year spans. E.g., «από τον 2ο μέχρι το τέλος του 6ου αι. μ.Χ.» (“from the 2nd until the end of the 6th c. AD”), «στη διάρκεια τοῦ 1ου αἰ. π.Χ» (“during the 1st c. BC”). Annotate entire period names. E.g., «νεότερη νεολιθική περίοδος» (“Late Neolithic period”). Period abbreviations are annotated as a single span (e.g., «ΥΕ ΙΙΙΒ/Γ» – “LH IIIB/C”). When a chronological period is followed by an explanatory date range in parentheses, the period and the explanation are annotated separately (e.g., «Πρωτοελλαδικής και Μεσοελλαδικής περιόδου» («3200/3000–1600 π.Χ.» – “Early Helladic and Middle Helladic periods” (“3200/3000–1600 BC”). Do not annotate indefinite periods, such as σε αὐτὴ τὴν περίοδο (this period).\n"
5,archaeology,SPECIES,"animal or plant species, including mythological creatures and biological remains","Animal and plant species. Ε.g., «Αγελάδα» προς τα δεξιά, θηλάζει το «μοσχαράκι» της (“Cow” to the right, suckling her “calf”). When a species (e.g., a shell) constitutes the material of an object, it is annotated as MATERIAL; for example, jewelry made of «Spondylus gaederopus». Mythological creatures such as gryphons are included under SPECIES. Mythological beings that speak or act as persons, such as the Sphinx, are annotated as PERSON. \n"
4,archaeology,MATERIAL,"raw material or substance used to make an object (e.g., marble, stone, metal, clay, bone)","Materials from which a find (ARTEFACT) or architectural member (CONTEXT) is made. Do not annotate adjectives defining materials. Ε.g., λευκό «μάρμαρο» (white “marble”); «αποκρουσμένο λίθο» (chipped “stone”). Do not annotate very general terms, e.g., αλλόχθονες πρώτες ύλες (allochthonous raw materials). Older architectural parts are annotated when reused as building materials (spolia), e.g., «αρχαία αρχιτεκτονικά μέλη» (“ancient architectural members”) used intact or with minimal hewing.\n"
2,archaeology,LOCATION,"specific geographic place, city, village, river, sea, or named archaeological site","cities, villages, locations, countries, rivers, seas, etc. Do not include very general concepts (e.g., οικισμός – settlement, αιγιαλίτιδα ζώνη – coastal zone). The term settlement is not annotated unless accompanied by a modifier (e.g., Ο ΠΡΟΪΣΤΟΡΙΚΟΣ «ΟΙΚΙΣΜΟΣ ΤΟΥ ΘΟΡΙΚΟΥ» – THE PREHISTORIC “SETTLEMENT OF THORIKOS”). Annotate places such as «Αρχαιολογικό Μουσείο Ιωαννίνων» (“Archaeological Museum of Ioannina”). Directions are not included in the annotation (e.g., βόρειο τμήμα του τεμένους – northern part of the sanctuary), nor are adjectives denoting origin (e.g., αιγύπτιου «ναοφόρου» – of an Egyptian “naophoros”). Pay attention to the meaning of the sentence. For example: “The scene of the Ascension of Alexander was particularly popular, from as early as the 10th century, in the Byzantine world and beyond”: here, «Βυζαντινό κόσμο» (“Byzantine world”) is tagged as LOCATION. In the phrase: «χάλκινες κοπὲς Κορίνθου» (“bronze mints of Corinth”), the reference is to Corinthian coinage (ARTΕFACT), not to the city of Corinth. Cities and countries are annotated as separate entities (e.g., «Darmstadt» «Γερμανίας» – “Darmstadt” in “Germany”). To avoid ambiguity arising from toponymic homonymy, villages are annotated together with their province or district as a single entity, e.g., «Ακρωτήρι Θήρας» (“Akrotiri of Thera”); «Ζάβαλι» «Λαδοχωρίου Ηγουμενίτσας» (“Zavali”, “Ladochori of Igoumenitsa”).\n"


### Data Preparation


In [ ]:
dataset_raw = ner_utils.prepare_gliner_dataset(df_texts, df_labels)

# Preview the first item
print(json.dumps(dataset_raw[0], indent=2, ensure_ascii=False))

2026-04-07 12:02:27 | INFO | Mapping 1 domains with 8 unique entity definitions.
2026-04-07 12:02:27 | INFO | Prepared 20 records for GLiNER2 inference.
{
  "id": 1,
  "domain": "archaeology",
  "text": "Κορινθιακό κιονόκρανο \nΑποκεκρουμένο ρωμαϊκό κιονόκρανο από λευκό μάρμαρο. Φέρει διακόσμηση από διπλή ζώνη ανάγλυφων φύλλων άκανθας και στο μέσο κάθε πλευράς του μη σωζόμενου άβακα άνθος με στέλεχος κινούμενο προς το μέσο μικρών ελίκων. \n",
  "starter_labels": [
    "ARTEFACT",
    "PERIOD",
    "LOCATION",
    "CONTEXT",
    "MATERIAL",
    "SPECIES",
    "PERSON",
    "FEATURE"
  ],
  "definitions": {
    "ARTEFACT": "archaeological find, portable object, vessel, tool, inscription, or detached architectural member",
    "PERIOD": "archaeological period, historical era, specific date, or century (e.g., Neolithic, 5th c. BC)",
    "LOCATION": "specific geographic place, city, village, river, sea, or named archaeological site",
    "CONTEXT": "immovable archaeological structure, in si

## Predict labels with GLiNER2

In [ ]:
# dataset_raw = dataset_raw[:1]

In [ ]:
df_predictions = ner_utils.process_ner_pipeline(dataset_raw, extractor)

if not df_predictions.empty:
    display(HTML("<h3>Pipeline Results Preview</h3>"))
    preview = df_predictions[['id', 'domain', 'summary']].head(10).style.set_properties(**{
        'text-align': 'left',
        'white-space': 'normal',
        'font-family': 'serif'
    })
    display(preview)



2026-03-20 16:31:14 | INFO | Running pipeline for 20 records...


Processing Entities:   0%|          | 0/20 [00:00<?, ?it/s]

2026-03-20 16:34:41 | INFO | Pipeline complete. Generated 20 processed rows.


,id,domain,summary
0,1,archaeology,"ρωμαϊκό κιονόκρανο [14:32] (ARTEFACT), άβακα άνθος [152:163] (CONTEXT)"
1,2,archaeology,"μοσχαράκι [77:86] (ARTEFACT), ΑΠΟΛ [293:297] (LOCATION), έδαφος [115:121] (CONTEXT), μοσχαράκι [77:86] (MATERIAL)"
2,3,archaeology,"κεφαλής Αθηνάς [30:44] (ARTEFACT), δίσκο [60:65] (LOCATION), λυχναριού [213:222] (MATERIAL), Αθηνάς [38:44] (PERSON)"
3,4,archaeology,"3ου αιώνα μ. Χ. [324:339] (PERIOD), 10ο αι. μ. Χ. [656:669] (PERIOD), 10ο αι. μ. Χ. [656:669] (PERIOD), Darmstadt της Γερμανίας [736:759] (LOCATION), γρύπες [215:221] (SPECIES)"
4,5,archaeology,
5,6,archaeology,"βυζαντινή περίοδο [219:236] (PERIOD), Μιχαήλ [30:36] (PERSON)"
6,7,archaeology,
7,8,archaeology,
8,9,archaeology,"κεφαλής νεαρού Διονύσου [101:124] (ARTEFACT), Όστρακο ερυθρόμορφου αγγείου [0:28] (CONTEXT)"
9,10,archaeology,"στοὰ [25:29] (ARTEFACT), ἀνατολικὴ πλευρὰ τοῦ γυμνασίου [235:265] (CONTEXT), στοὰ [25:29] (MATERIAL)"


In [ ]:
df_predictions

,id,domain,text,predictions,entity_count,offsets,summary
0,1,archaeology,Αποκεκρουμένο ρωμαϊκό κιονόκρανο από λευκό μάρμαρο. Φέρει διακόσμηση από διπλή ζώνη ανάγλυφων φύλλων άκανθας και στο μέσο κάθε πλευράς του μη σωζόμενου άβακα άνθος με στέλεχος κινούμενο προς το μέσο μικρών ελίκων. \n,"[{'start': 14, 'end': 32, 'label': 'ARTEFACT', 'text': 'ρωμαϊκό κιονόκρανο', 'confidence': None}, {'start': 152, 'end': 163, 'label': 'CONTEXT', 'text': 'άβακα άνθος', 'confidence': None}]",2,"[(14, 32, ARTEFACT), (152, 163, CONTEXT)]","ρωμαϊκό κιονόκρανο [14:32] (ARTEFACT), άβακα άνθος [152:163] (CONTEXT)"
1,2,archaeology,"Ασημένια δραχμή Απολλωνίας. Εμπροσθότυπος: Αγελάδα προς τα δεξιά, θηλάζει το μοσχαράκι της. Συμβατικά δηλώνεται το έδαφος με μια απλή γραμμή. Πάνω από την αγελάδα η επιγραφή ""ΔΟΝΑΞ"". Οπισθότυπος: Διπλό αστρικό κόσμημα σε οριζόντια διάταξη μέσα σε διπλό τετράγωνο πλαίσιο. Το εθνικό της πόλης ""ΑΠΟΛ"" πάνω από την κεντρική παράσταση. Στις άλλες τρεις πλευρές υπάρχει η επιγραφή ΜΟ-ΣΧ-ΟΥ σε κυκλική διάταξη","[{'start': 77, 'end': 86, 'label': 'ARTEFACT', 'text': 'μοσχαράκι', 'confidence': None}, {'start': 293, 'end': 297, 'label': 'LOCATION', 'text': 'ΑΠΟΛ', 'confidence': None}, {'start': 115, 'end': 121, 'label': 'CONTEXT', 'text': 'έδαφος', 'confidence': None}, {'start': 77, 'end': 86, 'label': 'MATERIAL', 'text': 'μοσχαράκι', 'confidence': None}]",4,"[(77, 86, ARTEFACT), (293, 297, LOCATION), (115, 121, CONTEXT), (77, 86, MATERIAL)]","μοσχαράκι [77:86] (ARTEFACT), ΑΠΟΛ [293:297] (LOCATION), έδαφος [115:121] (CONTEXT), μοσχαράκι [77:86] (MATERIAL)"
2,3,archaeology,"Λύχνος με ανάγλυφη διακόσμηση κεφαλής Αθηνάς (Προμάχου) στο δίσκο. Η θεά εικονίζεται σε κατατομή μέσα σε στεφάνη από εγχάρακτα γλωσσοειδή κοσμήματα. Στη βάση η υπογραφή ΡΗΓΛΟΥ, δηλώνει το εργαστήριο παραγωγής του λυχναριού.\n","[{'start': 30, 'end': 44, 'label': 'ARTEFACT', 'text': 'κεφαλής Αθηνάς', 'confidence': None}, {'start': 60, 'end': 65, 'label': 'LOCATION', 'text': 'δίσκο', 'confidence': None}, {'start': 213, 'end': 222, 'label': 'MATERIAL', 'text': 'λυχναριού', 'confidence': None}, {'start': 38, 'end': 44, 'label': 'PERSON', 'text': 'Αθηνάς', 'confidence': None}]",4,"[(30, 44, ARTEFACT), (60, 65, LOCATION), (213, 222, MATERIAL), (38, 44, PERSON)]","κεφαλής Αθηνάς [30:44] (ARTEFACT), δίσκο [60:65] (LOCATION), λυχναριού [213:222] (MATERIAL), Αθηνάς [38:44] (PERSON)"
3,4,archaeology,"Μαρμάρινη ορθογώνια πλάκα με ανάγλυφη παράσταση της Ανάληψης του Μεγάλου Αλεξάνδρου. Στο κέντρο απεικονίζεται ο Αλέξανδρος μετωπικός μέσα σε άρμα σχήματος καλαθιού. Στις δύο πλευρές αυτού, σε απόλυτη συμμετρία, δύο γρύπες σέρνουν το άρμα. Η παράσταση εμπνέεται από τη Μυθιστορία του Αλεξάνδρου, έργο του Ψευδοκαλλισθένη του 3ου αιώνα μ. Χ., το οποίο γνώρισε τεράστια διάδοση κατά τη διάρκεια του Μεσαίωνα. Στο κείμενο αυτό περιγράφονται πραγματικά ιστορικά γεγονότα συνδυασμένα με θρύλους, δοξασίες και μύθους, αναφορικά με το βίο και την πορεία του Αλεξάνδρου. Αν και η πρωιμότερη σωζόμενη απεικόνιση της Ανάληψης του Αλεξάνδρου μπορεί να χρονολογηθεί το 10ο αι. μ. Χ.-πρόκειται για ένα πλακίδιο κιβωτιδίου από ελεφαντοστό, σήμερα στο Darmstadt της Γερμανίας- εικονογραφικά φαίνεται να αντλεί τα πρότυπά της από τις ιδιαίτερα διαδεδομένες παραστάσεις αποθέωσης επιφανών ανδρών, ηρώων ή και αυτοκρατόρων της ελληνικής και ρωμαϊκής αρχαιότητας. Η σκηνή της Ανάληψης του Αλεξάνδρου ήταν ιδιαίτερα δημοφιλής, ήδη από τον 10ο αιώνα, στο Βυζαντινό και όχι μόνο κόσμο.","[{'start': 324, 'end': 339, 'label': 'PERIOD', 'text': '3ου αιώνα μ. Χ.', 'confidence': None}, {'start': 656, 'end': 669, 'label': 'PERIOD', 'text': '10ο αι. μ. Χ.', 'confidence': None}, {'start': 656, 'end': 669, 'label': 'PERIOD', 'text': '10ο αι. μ. Χ.', 'confidence': None}, {'start': 736, 'end': 759, 'label': 'LOCATION', 'text': 'Darmstadt της Γερμανίας', 'confidence': None}, {'start': 215, 'end': 221, 'label': 'SPECIES', 'text': 'γρύπες', 'confidence': None}]",5,"[(324, 339, PERIOD), (656, 669, PERIOD), (656, 669, PERIOD), (73

In [ ]:
df_entities_only = df_predictions.copy()
df_entities_only['entities'] = df_entities_only['predictions'].apply(lambda x: x.get('entities', {}))

# Drop the original 'predictions' and 'summary' columns if they exist
df_entities_only = df_entities_only.drop(columns=['predictions', 'summary'], errors='ignore')

display(HTML("<h3>GLiNER2 Entity Predictions Only</h3>"))
display(df_entities_only.head())

### Push to the Argilla annotation environment

In [ ]:
client = ner_utils.get_argilla_client()

DATASET_NAME = "genai-2026" # @param {type:"string"}
WORKSPACE_NAME = "genai-2026" # @param {type:"string"}

# running the code below overwrites the dataset in argilla
# _ = ner_utils.deploy_to_argilla(
#     client=client,
#     df_predictions=df_predictions,
#     df_labels=df_labels,
#     dataset_name=DATASET_NAME,
#     workspace_name=WORKSPACE_NAME
# );


2026-03-03 22:31:45 | INFO | Argilla: Logged in as argilla-courses with the role Role.owner


In [ ]:
print(f"Retrieving schema for dataset '{DATASET_NAME}' in workspace '{WORKSPACE_NAME}'...")

dataset = client.datasets.find(name=DATASET_NAME, workspace=WORKSPACE_NAME)

if dataset:
    print("\n--- Dataset Fields ---")
    if dataset.fields:
        for field in dataset.fields:
            print(f"  Name: {field.name}, Title: {field.title}, Type: {field.settings['type']}")
    else:
        print("  No fields defined for this dataset.")

    print("\n--- Dataset Questions (Labels) ---")
    if dataset.questions:
        for question in dataset.questions:
            print(f"  Name: {question.name}, Title: {question.title}, Type: {question.settings['type']}")
            if 'options' in question.settings:
                options = ', '.join([opt.value for opt in question.settings['options']])
                print(f"    Options: {options}")
            if 'labels' in question.settings:
                labels = ', '.join([label.value for label in question.settings['labels']])
                print(f"    Labels: {labels}")
    else:
        print("  No questions/labels defined for this dataset.")
else:
    print(f"Dataset '{DATASET_NAME}' not found in workspace '{WORKSPACE_NAME}'.")

## Evaluation Metrics for NER

- Evaluating a NER model is essential to measure its ability to accurately
identify and classify entities.
- The evaluation metrics typically focus on Precision, Recall, and F1-Score
- Calculated based on the comparison between the predicted entities and the actual entities in the dataset.




### Precision

- Precision measures the proportion of entities predicted by the model that are correct.
- High precision indicates that the model makes fewer false positive errors.

- $Precision = \frac{TP}{TP + FP}$



### Recall

- Recall measures the proportion of actual entities that are correctly identified by the model.
- High recall indicates that the model successfully captures most of the relevant entities.
- $Recall = \frac{TP}{TP + FN} $



### F1-Score

- The F1-Score is the harmonic mean of Precision and Recall, providing a
single score that balances the two.
- High F1-Score suggests a good
balance between precision and recall.


- $F_1 = 2 \times \frac{Precision \times Recall}{Precision + Recall} $


## Evaluation example

- Consider the following example:

`Apple Inc. is planning to open a new office in San Francisco in March 2025.`



#### Ground Truth (Actual Entities):


|               |              |
| ------------- | ------------ |
| Apple Inc.    | ORGANIZATION |
| San Francisco | LOCATION     |
| March 2025    | DATE         |



#### Model Prediction:


|               |                                  |
| ------------- | -------------------------------- |
| Apple Inc.    | ORGANIZATION ✅ (True Positive)  |
| San Francisco | LOCATION ✅ (True Positive)      |
| office        | LOCATION ❌ (False Positive)     |
| March 2025    | Not Detected ❌ (False Negative) |



#### Calculation:


|                      |                               |
| -------------------- | ----------------------------- |
| True Positives (TP)  | 2 (Apple Inc., San Francisco) |
| False Positives (FP) | 1 (office)                    |
| False Negatives (FN) | 1 (March 2025)                |

- $Precision = \frac{TP}{TP + FP} = \frac{2}{2 + 1} = 0.67$
- $Recall = \frac{TP}{TP + FN} =  \frac{2}{2 + 1} = 0.67$
- $F_1 = 2 \times \frac{Precision \times Recall}{Precision + Recall} = 2 \times \frac{0.67 \times 0.67}{0.67 + 0.67} = 0.67$
